<a href="https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [10]:
import pandas as pd
import os

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

# Create the action queue
playbook = df[[
    "content_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]].copy()

# Create a simple prioritization score
playbook["score"] = (
    (playbook["content_age_days"] > 180).astype(int)
    + (playbook["days_since_last_update"] > 180).astype(int)
    + (playbook["impressions_90d"] > 1000).astype(int)
    + (playbook["ctr"] < 0.10).astype(int)
    + (playbook["avg_position"] > 20).astype(int)
)

# Default reason
playbook["reason_code"] = "REVIEW"

# Assign reason codes
playbook.loc[
    (playbook["content_age_days"] > 180) &
    (playbook["days_since_last_update"] > 180) &
    (playbook["impressions_90d"] > 1000),
    "reason_code"
] = "STALE_HIGH_VISIBILITY"

playbook.loc[
    (playbook["ctr"] < 0.10) &
    (playbook["impressions_90d"] > 1000),
    "reason_code"
] = "LOW_CTR"

playbook.loc[
    playbook["avg_position"] > 20,
    "reason_code"
] = "POOR_POSITION"

# Assign actions
playbook["action"] = "HUMAN_REVIEW"

playbook.loc[
    playbook["reason_code"] == "STALE_HIGH_VISIBILITY",
    "action"
] = "CONTENT_REFRESH_REVIEW"

playbook.loc[
    playbook["reason_code"] == "LOW_CTR",
    "action"
] = "TITLE_SNIPPET_REVIEW"

playbook.loc[
    playbook["reason_code"] == "POOR_POSITION",
    "action"
] = "CONTENT_QUALITY_REVIEW"

# Rank the queue
playbook = playbook.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

playbook["rank"] = range(1, len(playbook) + 1)

display(playbook.head(20))

Dataset shape: (30000, 44)


,content_id,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,score,reason_code,action,rank
0,content_5feee3994adb,keyword article,231,194,7812,39.0,0.01,3590.0,5,POOR_POSITION,CONTENT_QUALITY_REVIEW,1
1,content_b16bd7307b39,keyword article,231,194,4590,31.0,0.00,4329.0,5,POOR_POSITION,CONTENT_QUALITY_REVIEW,2
2,content_66b4046cc144,keyword article,225,20,217415,26.6,0.03,5502.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,3
3,content_05e9b4cd9ccf,keyword article,299,104,179002,22.1,0.08,6974.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,4
4,content_0e70a832cb7a,keyword article,445,25,173450,22.6,0.04,3008.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,5
5,content_8b36799b7e44,keyword article,299,104,141400,32.0,0.02,7394.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,6
6,content_88d367c507a3,keyword article,333,104,130932,40.1,0.04,1476.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,7
7,content_e752a4e03dd3,keyword article,287,104,130892,23.9,0.01,6442.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,8
8,content_54baba704595,keyword article,286,104,130617,47.0,0.01,5813.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,9
9,content_124763d39ca5,keyword article,286,104,129803,33.2,0.01,5654.0,4,POOR_POSITION,CONTENT_QUALITY_REVIEW,10


## 1. Ranked actions + reason codes

The ranked queue is designed to prioritize pages for human review.

Pages with stronger observable signals are placed higher in the queue. The main actions are content refresh review, title/snippet review, and content quality review.

Reason codes:

- STALE_HIGH_VISIBILITY: The page is relatively old, has not been updated recently, and has meaningful impressions.
- LOW_CTR: The page receives impressions but has relatively low click-through rate.
- POOR_POSITION: The page has a relatively weak average search position.
- REVIEW: The available signals do not clearly support a specific action.

The score is a prioritization signal, not a guarantee that a page needs a particular action.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [11]:
print("Number of pages in action queue:", len(playbook))
print("Number of actions:", playbook["action"].nunique())

display(
    playbook["action"].value_counts().to_frame("count")
)

Number of pages in action queue: 30000
Number of actions: 4


,count
action,
HUMAN_REVIEW,19271
CONTENT_QUALITY_REVIEW,8539
TITLE_SNIPPET_REVIEW,2184
CONTENT_REFRESH_REVIEW,6


## 2. Intended use and limits

The action queue is intended to help a content team prioritize which pages should be reviewed first.

It can help focus limited review time on pages with stronger observable signals. The queue is decision-support rather than an automated production system.

The score does not prove that a page is declining, and it does not prove that refreshing a page will improve its search performance.

The recommendations are based only on the available dataset and may not generalize to different clients, time periods, or content types.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [12]:
# Show the distribution of recommended actions
action_summary = (
    playbook["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

display(action_summary)

,action,count
0,HUMAN_REVIEW,19271
1,CONTENT_QUALITY_REVIEW,8539
2,TITLE_SNIPPET_REVIEW,2184
3,CONTENT_REFRESH_REVIEW,6


## 3. Human review + the no-go list

Every recommended action requires human review before implementation.

A reviewer should check whether the page is relevant, accurate, useful, and appropriate for the recommended action. The reviewer should also consider information that is not represented in the dataset.

The following should NOT be automated:

- Publishing or rewriting content automatically.
- Deleting pages automatically.
- Changing titles or metadata without human review.
- Making claims about search-engine causality.
- Guaranteeing that a refresh will improve performance.
- Making irreversible content decisions.

The model output should remain a prioritization aid for human decision-making.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [13]:
# Basic monitoring information for the current queue

print("Current queue size:", len(playbook))
print("Average score:", round(playbook["score"].mean(), 3))
print("Maximum score:", playbook["score"].max())

print("\nAction distribution:")
display(playbook["action"].value_counts())

Current queue size: 30000
Average score: 1.867
Maximum score: 5

Action distribution:


,count
action,
HUMAN_REVIEW,19271
CONTENT_QUALITY_REVIEW,8539
TITLE_SNIPPET_REVIEW,2184
CONTENT_REFRESH_REVIEW,6


## 4. Monitoring / retrain triggers

The recommendations should be monitored over time using the same evaluation approach used during model development.

A review or retraining cycle should be considered if:

- Precision@50 decreases materially on a new evaluation period.
- The distribution of important input features changes substantially.
- New clients or content types are introduced.
- The relationship between model scores and observed declining pages changes.
- Human reviewers repeatedly disagree with the recommended actions.

These triggers indicate that the model or action rules may need to be reviewed. They do not by themselves prove that retraining is required.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [14]:
import os

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Export the ranked queue
output_path = "work/outputs/action_playbook_queue.csv"

playbook.to_csv(
    output_path,
    index=False
)

print("Export completed successfully.")
print("File:", output_path)
print("Rows exported:", len(playbook))
print("Columns exported:", len(playbook.columns))

Export completed successfully.
File: work/outputs/action_playbook_queue.csv
Rows exported: 30000
Columns exported: 12


## 5. Exports for the paper

The ranked action queue is exported to `work/outputs/` so that it can be reused as an input to the research paper.

The exported queue contains the page identifier, observable signals, prioritization score, reason code, recommended action, and rank.

The queue is regenerated by the notebook rather than treated as a permanent production dataset.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.